# NVARC Direct-Sampling Kaggle Proxy

This notebook measures frozen NVARC decoding under the same two-attempt protocol planned for Diffusion-ARC.

**Primary question:** how well does the released NVARC/Qwen3-4B checkpoint perform when each canonical ARC test input receives at most two direct autoregressive samples?

The measured path deliberately excludes test-time training, task/input augmentations, DFS or beam search, candidate pools, rescoring, ranking, voting, and invalid-sample retries. A sample is decoded without target-shape information. Public labels are loaded only after inference for evaluation metrics.

Success means producing a complete Kaggle `submission.json`, preserving the predeclared attempt/seed mapping, and recording enough configuration, coverage, validity, and runtime information to reproduce and interpret the score.


## Required Kaggle inputs

Use four NVIDIA L4 GPUs with Internet disabled and attach:

- Competition: `arc-prize-2026-arc-agi-2`
- Model: `sorokin/qwen3_4b_grids15_sft139/Transformers/bfloat16/1`
- Utility script: `sorokin/pip-install-unsloth-flash-patch`

The companion `kernel-metadata.json` declares these inputs. The utility script provides the Unsloth and Flash Attention environment used by the existing NVARC Kaggle runs. This notebook does not download models, datasets, packages, or wheels.

A competition rerun always switches to `submission` mode and processes every hidden task. For an interactive public smoke test, leave `RUN_MODE = "evaluation"` and start with a small `EVALUATION_TASK_LIMIT`.


In [ ]:
# User configuration: normally only RUN_VARIANT and the smoke-test limit change.
from __future__ import annotations

import json
import os
import random
import time
from pathlib import Path

AVAILABLE_VARIANTS = {
    "S1": ("A", "A"),
    "S2_A": ("A", "B"),
    "S2_B": ("C", "D"),
    "S2_C": ("E", "F"),
}
RUN_VARIANT = "S1"  # Choose one of: S1, S2_A, S2_B, S2_C.

RUN_MODE = "evaluation"  # "evaluation" or "submission"; reruns force submission.
EVALUATION_TASK_LIMIT = 4  # Set to None for all 120 public evaluation tasks.

MODEL_PATH = "/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1"
COMPETITION_DATA_DIR = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"

NUM_GPUS = 4
EXPECTED_GPU_SUBSTRING = "L4"
MAX_SEQ_LENGTH = 8192
RANDOM_SEED = 42
SEED_NAMESPACE = "nvarc-direct-sampling-v1"
SAMPLING_TEMPERATURE = 1.0
TASK_TIME_LIMIT_SECONDS = 1200
NOTEBOOK_TIME_LIMIT_SECONDS = 12 * 3600
SHUTDOWN_MARGIN_SECONDS = 600
INVALID_SUBMISSION_PLACEHOLDER = [[0]]

assert RUN_VARIANT in AVAILABLE_VARIANTS, (RUN_VARIANT, tuple(AVAILABLE_VARIANTS))
assert RUN_MODE in {"evaluation", "submission"}
assert EVALUATION_TASK_LIMIT is None or EVALUATION_TASK_LIMIT > 0
assert SAMPLING_TEMPERATURE > 0
assert INVALID_SUBMISSION_PLACEHOLDER == [[0]]


def env_flag(name: str) -> bool:
    return os.getenv(name, "").strip().lower() in {"1", "true", "yes", "y"}


competition_rerun = env_flag("KAGGLE_IS_COMPETITION_RERUN")
effective_run_mode = "submission" if competition_rerun else RUN_MODE
data_dir = Path(COMPETITION_DATA_DIR)
challenge_path = data_dir / (
    "arc-agi_test_challenges.json"
    if effective_run_mode == "submission"
    else "arc-agi_evaluation_challenges.json"
)
solutions_path = (
    None
    if effective_run_mode == "submission"
    else data_dir / "arc-agi_evaluation_solutions.json"
)

run_id = time.strftime("%Y%m%d-%H%M%S") + f"-{effective_run_mode}-{RUN_VARIANT.lower()}"
run_dir = Path("/kaggle/working/nvarc_direct_sampling_runs") / run_id
run_dir.mkdir(parents=True, exist_ok=False)

random.seed(RANDOM_SEED)
global_deadline = time.time() + NOTEBOOK_TIME_LIMIT_SECONDS - SHUTDOWN_MARGIN_SECONDS
attempt_seed_labels = list(AVAILABLE_VARIANTS[RUN_VARIANT])

protocol = {
    "canonical_input_only": True,
    "direct_ancestral_sampling": True,
    "attempts_per_output": 2,
    "unique_draws_per_output": len(set(attempt_seed_labels)),
    "test_time_training": False,
    "test_time_augmentation": False,
    "dfs_or_beam_search": False,
    "candidate_pool": False,
    "rescoring_or_ranking": False,
    "voting": False,
    "invalid_sample_retry": False,
    "target_shape_available_to_decoder": False,
}
assert protocol["unique_draws_per_output"] == (1 if RUN_VARIANT == "S1" else 2)
assert not any(
    protocol[name]
    for name in (
        "test_time_training",
        "test_time_augmentation",
        "dfs_or_beam_search",
        "candidate_pool",
        "rescoring_or_ranking",
        "voting",
        "invalid_sample_retry",
        "target_shape_available_to_decoder",
    )
)

config = {
    "requested_run_mode": RUN_MODE,
    "run_mode": effective_run_mode,
    "competition_rerun": competition_rerun,
    "evaluation_task_limit": EVALUATION_TASK_LIMIT,
    "run_variant": RUN_VARIANT,
    "available_variants": {name: list(labels) for name, labels in AVAILABLE_VARIANTS.items()},
    "attempt_seed_labels": attempt_seed_labels,
    "model_path": MODEL_PATH,
    "competition_data_dir": str(data_dir),
    "challenge_path": str(challenge_path),
    "solutions_path": None if solutions_path is None else str(solutions_path),
    "run_id": run_id,
    "run_dir": str(run_dir),
    "num_gpus": NUM_GPUS,
    "expected_gpu_substring": EXPECTED_GPU_SUBSTRING,
    "max_seq_length": MAX_SEQ_LENGTH,
    "seed": RANDOM_SEED,
    "seed_namespace": SEED_NAMESPACE,
    "sampling_temperature": SAMPLING_TEMPERATURE,
    "task_time_limit_seconds": TASK_TIME_LIMIT_SECONDS,
    "global_deadline": global_deadline,
    "invalid_submission_placeholder": INVALID_SUBMISSION_PLACEHOLDER,
    "protocol": protocol,
}
config_path = run_dir / "config.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
config


In [ ]:
# Fail early when a declared Kaggle input or utility-provided package is absent.
import importlib.metadata
import platform

assert Path(config["model_path"]).is_dir(), config["model_path"]
assert Path(config["challenge_path"]).is_file(), config["challenge_path"]
if config["solutions_path"] is not None:
    assert Path(config["solutions_path"]).is_file(), config["solutions_path"]

required_distributions = ("torch", "transformers", "unsloth", "unsloth_zoo", "flash_attn")
package_versions = {}
missing_distributions = []
for distribution in required_distributions:
    try:
        package_versions[distribution] = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        missing_distributions.append(distribution)
assert not missing_distributions, (
    "Attach the sorokin/pip-install-unsloth-flash-patch utility script; "
    f"missing distributions: {missing_distributions}"
)

environment = {
    "python": platform.python_version(),
    "packages": package_versions,
    "model_path": config["model_path"],
    "challenge_path": config["challenge_path"],
}
(run_dir / "environment.json").write_text(
    json.dumps(environment, indent=2), encoding="utf-8"
)
environment


## Direct-sampling implementation

The module below owns canonical ARC serialization, stable per-output seeds, constrained ancestral token sampling, parsing, worker-side result persistence, Kaggle submission construction, and public evaluation metrics.

`S1` executes only draw A and copies it into both attempt fields. The three pass@2 variants execute exactly two distinct predeclared draws. Invalid or incomplete generations are not retried; `[[0]]` is used only to keep the submission syntactically complete, while public metrics explicitly count that attempt as invalid.


In [ ]:
%%writefile nvarc_direct_sampling.py
from __future__ import annotations

import bz2
import hashlib
import importlib.metadata
import json
import os
import pickle
import random
import time
import traceback
from pathlib import Path

import numpy as np
import torch
from transformers import AutoTokenizer


ARC_TOKENS = list(range(11)) + [15]
EOS_ID = 15


def stable_seed(*parts: object) -> int:
    payload = "|".join(map(str, parts)).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "big") % (2**31)


def convert_grid_to_string(grid: list[list[int]] | np.ndarray) -> str:
    return "\n".join("".join(str(int(cell)) for cell in row) for row in grid)


def is_valid_grid(grid: object) -> bool:
    if not isinstance(grid, (list, np.ndarray)):
        return False
    array = np.asarray(grid)
    if array.ndim != 2 or not all(1 <= size <= 30 for size in array.shape):
        return False
    return bool(np.issubdtype(array.dtype, np.integer) and np.all((0 <= array) & (array <= 9)))


def copy_grid(grid: list[list[int]]) -> list[list[int]]:
    return [[int(cell) for cell in row] for row in grid]


class QwenFormatter:
    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    @staticmethod
    def format_query(test_input: list[list[int]]) -> str:
        grid_input = convert_grid_to_string(test_input)
        return f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n"

    @staticmethod
    def format_demonstrations(train: list[dict]) -> str:
        text = ""
        for example in train:
            grid_input = convert_grid_to_string(example["input"])
            grid_output = convert_grid_to_string(example["output"])
            text += (
                f"<|im_start|>user\n{grid_input}<|im_end|>"
                f"<|im_start|>assistant\n{grid_output}<|im_end|>"
            )
        return text

    def format_prompt(self, train: list[dict], test_input: list[list[int]]) -> str:
        return self.format_demonstrations(train) + self.format_query(test_input)

    def max_new_tokens(self) -> int:
        largest_grid = np.zeros((30, 30), dtype=np.int8)
        reply = convert_grid_to_string(largest_grid) + "<|im_end|>"
        return len(self.tokenizer.encode(reply)) + 1

    def convert_tokens_to_grid(self, tokens: list[int]) -> list[list[int]] | None:
        if not tokens or tokens[-1] != EOS_ID:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        lines = text.strip().splitlines()
        if not lines or len(lines) > 30:
            return None
        rows = []
        for line in lines:
            line = line.strip()
            if not line or len(line) > 30 or any(char not in "0123456789" for char in line):
                return None
            rows.append([int(char) for char in line])
        if len({len(row) for row in rows}) != 1 or not is_valid_grid(rows):
            return None
        return rows


def prepare_canonical_prompt(
    task: dict,
    test_index: int,
    formatter: QwenFormatter,
    max_prompt_tokens: int,
) -> dict:
    train = list(task["train"])
    test_input = task["test"][test_index]["input"]
    dropped_demonstrations = 0
    while True:
        prompt = formatter.format_prompt(train, test_input)
        input_ids = formatter.tokenizer.encode(prompt)
        if len(input_ids) <= max_prompt_tokens:
            return {
                "prompt": prompt,
                "input_ids": input_ids,
                "dropped_demonstrations": dropped_demonstrations,
                "retained_demonstrations": len(train),
            }
        if not train:
            raise ValueError(
                f"test input {test_index} exceeds the {max_prompt_tokens}-token prompt limit"
            )
        train = train[1:]
        dropped_demonstrations += 1


def verify_checkpoint_contract(model, tokenizer) -> dict:
    for token, expected_id in [(str(index), index) for index in range(10)]:
        actual_id = tokenizer.convert_tokens_to_ids(token)
        assert actual_id == expected_id, (token, actual_id, expected_id)
    assert tokenizer.convert_tokens_to_ids("Ċ") == 10
    assert tokenizer.convert_tokens_to_ids("<|im_end|>") == EOS_ID
    embedding_rows = int(model.get_input_embeddings().weight.shape[0])
    output_rows = int(model.get_output_embeddings().weight.shape[0])
    assert embedding_rows == output_rows == 16, (embedding_rows, output_rows)
    return {
        "embedding_rows": embedding_rows,
        "output_rows": output_rows,
        "parameter_dtypes": sorted({str(parameter.dtype) for parameter in model.parameters()}),
    }


def load_frozen_model(config: dict):
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=config["model_path"],
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=config["max_seq_length"],
        dtype=torch.bfloat16,
    )
    model = FastLanguageModel.for_inference(model)
    model.requires_grad_(False)
    model.eval()
    assert not any(parameter.requires_grad for parameter in model.parameters())
    return model, tokenizer, verify_checkpoint_contract(model, tokenizer)


@torch.inference_mode()
def autoregressive_sample(
    model,
    prefix_tokens: list[int],
    max_new_tokens: int,
    temperature: float,
    seed: int,
    deadline: float,
) -> dict:
    device = next(model.parameters()).device
    input_ids = torch.tensor([prefix_tokens], device=device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    cache = outputs.past_key_values
    logits = outputs.logits[0, -1]
    generated = []
    sampling_nll = 0.0
    generator = torch.Generator(device=device)
    generator.manual_seed(seed)
    stop_reason = "max_new_tokens"

    for position_offset in range(max_new_tokens):
        if time.time() >= deadline:
            stop_reason = "deadline"
            break
        allowed_logits = logits[ARC_TOKENS].float() / temperature
        allowed_log_probs = allowed_logits.log_softmax(-1)
        relative_index = int(
            torch.multinomial(
                allowed_log_probs.exp(),
                num_samples=1,
                generator=generator,
            ).item()
        )
        token = ARC_TOKENS[relative_index]
        sampling_nll -= float(allowed_log_probs[relative_index].item())
        generated.append(token)
        if token == EOS_ID:
            stop_reason = "eos"
            break
        outputs = model(
            input_ids=torch.tensor([[token]], device=device, dtype=torch.long),
            position_ids=torch.tensor(
                [[len(prefix_tokens) + position_offset]],
                device=device,
                dtype=torch.long,
            ),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )
        cache = outputs.past_key_values
        logits = outputs.logits[0, -1]

    return {
        "tokens": generated,
        "sampling_nll": sampling_nll,
        "stop_reason": stop_reason,
        "ended_with_eos": bool(generated and generated[-1] == EOS_ID),
    }


def failed_output_record(config: dict, test_index: int, error: str) -> dict:
    placeholder = copy_grid(config["invalid_submission_placeholder"])
    labels = list(config["attempt_seed_labels"])
    return {
        "test_index": test_index,
        "attempt_seed_labels": labels,
        "attempts": [copy_grid(placeholder), copy_grid(placeholder)],
        "attempt_valid": [False, False],
        "samples": {},
        "prompt_tokens": None,
        "dropped_demonstrations": None,
        "retained_demonstrations": None,
        "elapsed_seconds": 0.0,
        "error": error,
    }


def solve_output(
    model,
    formatter: QwenFormatter,
    task_id: str,
    task: dict,
    test_index: int,
    config: dict,
    deadline: float,
) -> dict:
    started = time.time()
    max_new_tokens = formatter.max_new_tokens()
    prepared = prepare_canonical_prompt(
        task,
        test_index,
        formatter,
        config["max_seq_length"] - max_new_tokens,
    )
    samples = {}
    for label in dict.fromkeys(config["attempt_seed_labels"]):
        seed = stable_seed(
            config["seed_namespace"],
            config["seed"],
            task_id,
            test_index,
            label,
        )
        sampled = autoregressive_sample(
            model=model,
            prefix_tokens=prepared["input_ids"],
            max_new_tokens=max_new_tokens,
            temperature=config["sampling_temperature"],
            seed=seed,
            deadline=deadline,
        )
        grid = formatter.convert_tokens_to_grid(sampled["tokens"])
        samples[label] = {
            "seed": seed,
            "sampling_nll": sampled["sampling_nll"],
            "stop_reason": sampled["stop_reason"],
            "ended_with_eos": sampled["ended_with_eos"],
            "generated_tokens": len(sampled["tokens"]),
            "valid": grid is not None,
            "grid": grid,
        }

    placeholder = config["invalid_submission_placeholder"]
    attempts = []
    attempt_valid = []
    for label in config["attempt_seed_labels"]:
        sample = samples[label]
        attempt_valid.append(bool(sample["valid"]))
        attempts.append(
            copy_grid(sample["grid"])
            if sample["valid"]
            else copy_grid(placeholder)
        )
    return {
        "test_index": test_index,
        "attempt_seed_labels": list(config["attempt_seed_labels"]),
        "attempts": attempts,
        "attempt_valid": attempt_valid,
        "samples": samples,
        "prompt_tokens": len(prepared["input_ids"]),
        "dropped_demonstrations": prepared["dropped_demonstrations"],
        "retained_demonstrations": prepared["retained_demonstrations"],
        "elapsed_seconds": time.time() - started,
        "error": None,
    }


def solve_task(model, tokenizer, task_id: str, task: dict, config: dict) -> dict:
    task_started = time.time()
    deadline = min(
        config["global_deadline"],
        task_started + config["task_time_limit_seconds"],
    )
    formatter = QwenFormatter(tokenizer)
    outputs = []
    for test_index in range(len(task["test"])):
        if time.time() >= deadline:
            outputs.append(failed_output_record(config, test_index, "deadline before output"))
            continue
        try:
            outputs.append(
                solve_output(
                    model,
                    formatter,
                    task_id,
                    task,
                    test_index,
                    config,
                    deadline,
                )
            )
        except Exception as error:
            outputs.append(failed_output_record(config, test_index, repr(error)))
    return {
        "task_id": task_id,
        "outputs": outputs,
        "expected_outputs": len(task["test"]),
        "elapsed_seconds": time.time() - task_started,
        "deadline_reached": time.time() >= deadline,
    }


def write_pickle_bz2(path: str | Path, value) -> None:
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with bz2.BZ2File(temporary, "wb") as handle:
        pickle.dump(value, handle, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(temporary, path)


def worker(rank: int, queue, config_path: str) -> None:
    with open(config_path, encoding="utf-8") as handle:
        config = json.load(handle)
    run_dir = Path(config["run_dir"])
    output_dir = run_dir / "task_results"
    error_dir = run_dir / "errors"
    output_dir.mkdir(parents=True, exist_ok=True)
    error_dir.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(config["seed"] + rank)
    np.random.seed(config["seed"] + rank)
    random.seed(config["seed"] + rank)
    model, tokenizer, checkpoint_contract = load_frozen_model(config)

    with open(config["challenge_path"], encoding="utf-8") as handle:
        challenges = json.load(handle)
    manifest = {
        "rank": rank,
        "gpu": torch.cuda.get_device_name(0),
        "torch": importlib.metadata.version("torch"),
        "transformers": importlib.metadata.version("transformers"),
        "unsloth": importlib.metadata.version("unsloth"),
        "checkpoint_contract": checkpoint_contract,
        "all_parameters_frozen": not any(
            parameter.requires_grad for parameter in model.parameters()
        ),
        "run_variant": config["run_variant"],
        "attempt_seed_labels": config["attempt_seed_labels"],
        "protocol": config["protocol"],
    }
    (run_dir / f"worker_{rank}_manifest.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )

    while True:
        if time.time() >= config["global_deadline"]:
            break
        task_id = queue.get()
        if task_id is None:
            break
        try:
            result = solve_task(model, tokenizer, task_id, challenges[task_id], config)
            write_pickle_bz2(output_dir / f"{task_id}.pbz2", result)
            print(f"[GPU {rank}] {task_id}: {result['elapsed_seconds']:.1f}s")
        except Exception as error:
            failure = {
                "task_id": task_id,
                "rank": rank,
                "error": repr(error),
                "traceback": traceback.format_exc(),
            }
            (error_dir / f"{task_id}.json").write_text(
                json.dumps(failure, indent=2), encoding="utf-8"
            )
            print(f"[GPU {rank}] ERROR {task_id}: {error!r}")


def load_task_results(run_dir: str | Path) -> dict[str, dict]:
    results = {}
    for path in sorted((Path(run_dir) / "task_results").glob("*.pbz2")):
        with bz2.BZ2File(path, "rb") as handle:
            result = pickle.load(handle)
        results[result["task_id"]] = result
    return results


def make_submission(challenges: dict, task_results: dict, placeholder: list[list[int]]) -> dict:
    submission = {
        task_id: [
            {"attempt_1": copy_grid(placeholder), "attempt_2": copy_grid(placeholder)}
            for _ in task["test"]
        ]
        for task_id, task in challenges.items()
    }
    for task_id, result in task_results.items():
        for output in result["outputs"]:
            target = submission[task_id][output["test_index"]]
            target["attempt_1"] = copy_grid(output["attempts"][0])
            target["attempt_2"] = copy_grid(output["attempts"][1])
    validate_submission(challenges, submission)
    return submission


def validate_submission(challenges: dict, submission: dict) -> None:
    assert set(submission) == set(challenges)
    for task_id, task in challenges.items():
        assert len(submission[task_id]) == len(task["test"])
        for attempts in submission[task_id]:
            assert set(attempts) == {"attempt_1", "attempt_2"}
            assert is_valid_grid(attempts["attempt_1"])
            assert is_valid_grid(attempts["attempt_2"])


def output_record(task_results: dict, task_id: str, test_index: int) -> dict | None:
    result = task_results.get(task_id)
    if result is None or test_index >= len(result["outputs"]):
        return None
    return result["outputs"][test_index]


def aggregate_diagnostics(task_ids: list[str], challenges: dict, task_results: dict) -> dict:
    diagnostics = {
        "expected_outputs": 0,
        "covered_outputs": 0,
        "generated_sequences": 0,
        "valid_sequences": 0,
        "invalid_sequences": 0,
        "valid_attempts": 0,
        "invalid_attempts": 0,
        "duplicate_attempt_outputs": 0,
        "valid_duplicate_attempt_outputs": 0,
        "outputs_with_dropped_demonstrations": 0,
        "dropped_demonstrations": 0,
        "output_errors": 0,
        "deadline_reached_task_files": 0,
        "runtime_seconds": 0.0,
    }
    for task_id in task_ids:
        result = task_results.get(task_id)
        diagnostics["expected_outputs"] += len(challenges[task_id]["test"])
        if result is None:
            continue
        diagnostics["deadline_reached_task_files"] += int(result["deadline_reached"])
        diagnostics["runtime_seconds"] += float(result["elapsed_seconds"])
        for output in result["outputs"]:
            diagnostics["covered_outputs"] += 1
            diagnostics["generated_sequences"] += len(output["samples"])
            valid_samples = sum(int(sample["valid"]) for sample in output["samples"].values())
            diagnostics["valid_sequences"] += valid_samples
            diagnostics["invalid_sequences"] += len(output["samples"]) - valid_samples
            valid_attempts = sum(map(int, output["attempt_valid"]))
            diagnostics["valid_attempts"] += valid_attempts
            diagnostics["invalid_attempts"] += 2 - valid_attempts
            duplicate = output["attempts"][0] == output["attempts"][1]
            diagnostics["duplicate_attempt_outputs"] += int(duplicate)
            diagnostics["valid_duplicate_attempt_outputs"] += int(
                duplicate and all(output["attempt_valid"])
            )
            dropped = output["dropped_demonstrations"] or 0
            diagnostics["outputs_with_dropped_demonstrations"] += int(dropped > 0)
            diagnostics["dropped_demonstrations"] += dropped
            diagnostics["output_errors"] += int(output["error"] is not None)
    return diagnostics


def evaluate_predictions(
    task_ids: list[str],
    challenges: dict,
    solutions: dict,
    task_results: dict,
) -> dict:
    total_outputs = 0
    pass1_outputs = 0
    pass2_outputs = 0
    task_averaged_pass1 = 0.0
    task_averaged_pass2 = 0.0
    strict_pass1_tasks = 0
    strict_pass2_tasks = 0
    wrong_shape_attempts = [0, 0]
    solved_outputs = []

    for task_id in task_ids:
        targets = solutions[task_id]
        task_pass1 = 0
        task_pass2 = 0
        for test_index, target in enumerate(targets):
            total_outputs += 1
            output = output_record(task_results, task_id, test_index)
            if output is None:
                continue
            target_array = np.asarray(target)
            predictions = [np.asarray(grid) for grid in output["attempts"]]
            valid = list(map(bool, output["attempt_valid"]))
            for attempt_index, prediction in enumerate(predictions):
                if valid[attempt_index] and prediction.shape != target_array.shape:
                    wrong_shape_attempts[attempt_index] += 1
            correct1 = valid[0] and np.array_equal(predictions[0], target_array)
            correct2 = correct1 or (
                valid[1] and np.array_equal(predictions[1], target_array)
            )
            task_pass1 += int(correct1)
            task_pass2 += int(correct2)
            pass1_outputs += int(correct1)
            pass2_outputs += int(correct2)
            if correct2:
                solved_outputs.append(f"{task_id}_{test_index}")
        output_count = len(targets)
        task_averaged_pass1 += task_pass1 / output_count
        task_averaged_pass2 += task_pass2 / output_count
        strict_pass1_tasks += int(task_pass1 == output_count)
        strict_pass2_tasks += int(task_pass2 == output_count)

    task_count = len(task_ids)
    assert task_count > 0 and total_outputs > 0
    return {
        "kaggle_output_weighted_pass_at_1": pass1_outputs / total_outputs,
        "kaggle_output_weighted_pass_at_2": pass2_outputs / total_outputs,
        "task_averaged_pass_at_1": task_averaged_pass1 / task_count,
        "task_averaged_pass_at_2": task_averaged_pass2 / task_count,
        "strict_fully_solved_tasks_at_1": strict_pass1_tasks,
        "strict_fully_solved_tasks_at_2": strict_pass2_tasks,
        "task_count": task_count,
        "output_count": total_outputs,
        "correct_outputs_at_1": pass1_outputs,
        "correct_outputs_at_2": pass2_outputs,
        "wrong_shape_valid_attempts": {
            "attempt_1": wrong_shape_attempts[0],
            "attempt_2": wrong_shape_attempts[1],
        },
        "outputs_with_pass_at_2": solved_outputs,
        "invalid_attempts_count_as_incorrect": True,
    }


def aggregate_run(config_path: str | Path) -> dict:
    config_path = Path(config_path)
    with config_path.open(encoding="utf-8") as handle:
        config = json.load(handle)
    run_dir = Path(config["run_dir"])
    with open(config["challenge_path"], encoding="utf-8") as handle:
        challenges = json.load(handle)
    requested_task_ids = sorted(challenges)
    if config["run_mode"] == "evaluation" and config["evaluation_task_limit"] is not None:
        requested_task_ids = requested_task_ids[: config["evaluation_task_limit"]]

    task_results = load_task_results(run_dir)
    placeholder = config["invalid_submission_placeholder"]
    submission = make_submission(challenges, task_results, placeholder)
    submission_path = Path("/kaggle/working/submission.json")
    submission_path.write_text(json.dumps(submission), encoding="utf-8")
    (run_dir / "submission.json").write_text(json.dumps(submission), encoding="utf-8")
    write_pickle_bz2(run_dir / "task_results.pbz2", task_results)

    error_files = sorted(path.name for path in (run_dir / "errors").glob("*.json"))
    missing_task_ids = sorted(set(requested_task_ids) - set(task_results))
    diagnostics = aggregate_diagnostics(
        requested_task_ids, challenges, task_results
    )
    summary = {
        "run_id": config["run_id"],
        "run_mode": config["run_mode"],
        "run_variant": config["run_variant"],
        "attempt_seed_labels": config["attempt_seed_labels"],
        "protocol": config["protocol"],
        "requested_tasks": len(requested_task_ids),
        "completed_task_files": len(set(requested_task_ids) & set(task_results)),
        "missing_task_ids": missing_task_ids,
        "error_files": error_files,
        "diagnostics": diagnostics,
        "submission_path": str(submission_path),
        "task_results_path": str(run_dir / "task_results.pbz2"),
        "placeholder_may_receive_kaggle_credit": True,
    }

    if config["run_mode"] == "evaluation":
        with open(config["solutions_path"], encoding="utf-8") as handle:
            solutions = json.load(handle)
        metrics = evaluate_predictions(
            requested_task_ids,
            challenges,
            solutions,
            task_results,
        )
        evaluation_results = {
            "config": config,
            "coverage": {
                "requested_tasks": len(requested_task_ids),
                "completed_task_files": summary["completed_task_files"],
                "missing_task_ids": missing_task_ids,
                "error_files": error_files,
            },
            "diagnostics": diagnostics,
            "metrics": metrics,
        }
        evaluation_path = run_dir / "evaluation_results.json"
        evaluation_path.write_text(
            json.dumps(evaluation_results, indent=2), encoding="utf-8"
        )
        summary["evaluation_results_path"] = str(evaluation_path)
        summary["metrics"] = metrics

    (run_dir / "run_summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    return summary


In [ ]:
%%writefile nvarc_direct_sampling_runner.py
from __future__ import annotations

import argparse
import json
import os
import time
from pathlib import Path

import torch
import torch.multiprocessing as mp


def local_worker(rank: int, queue, config_path: str) -> None:
    with open(config_path, encoding="utf-8") as handle:
        config = json.load(handle)
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
    os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
    os.environ["OMP_NUM_THREADS"] = "12"
    torch.set_default_device("cpu")

    marker_dir = Path(config["run_dir"]) / "worker_markers"
    marker_dir.mkdir(parents=True, exist_ok=True)
    if rank > 0:
        previous_marker = marker_dir / f"worker_{rank - 1}.ready"
        while not previous_marker.exists():
            if time.time() >= config["global_deadline"]:
                raise TimeoutError(f"Timed out waiting for {previous_marker}")
            time.sleep(2)

    # Unsloth's runtime patching is not process-safe, so imports are serialized.
    from unsloth import FastLanguageModel  # noqa: F401

    (marker_dir / f"worker_{rank}.ready").write_text("ready", encoding="utf-8")
    from nvarc_direct_sampling import worker

    worker(rank, queue, config_path)


def main(config_path: str) -> None:
    with open(config_path, encoding="utf-8") as handle:
        config = json.load(handle)
    assert torch.cuda.is_available(), "CUDA is required"
    assert torch.cuda.device_count() >= config["num_gpus"], (
        torch.cuda.device_count(),
        config["num_gpus"],
    )
    gpu_names = [
        torch.cuda.get_device_name(index) for index in range(config["num_gpus"])
    ]
    expected = config.get("expected_gpu_substring")
    if expected:
        assert all(expected in name for name in gpu_names), gpu_names

    with open(config["challenge_path"], encoding="utf-8") as handle:
        task_ids = sorted(json.load(handle))
    if config["run_mode"] == "evaluation" and config["evaluation_task_limit"] is not None:
        task_ids = task_ids[: config["evaluation_task_limit"]]

    manager = mp.Manager()
    queue = manager.Queue()
    for task_id in task_ids:
        queue.put(task_id)
    for _ in range(config["num_gpus"]):
        queue.put(None)

    print(
        f"Running {len(task_ids)} tasks with variant {config['run_variant']} "
        f"on {config['num_gpus']} GPUs: {gpu_names}"
    )
    mp.spawn(
        local_worker,
        args=(queue, config_path),
        nprocs=config["num_gpus"],
        join=True,
    )


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    arguments = parser.parse_args()
    main(arguments.config)


## Run inference

Every worker loads one frozen BF16 NVARC copy onto its assigned L4. Tasks are distributed through a shared queue, but sample seeds depend only on the fixed namespace, global seed, task ID, test index, and draw label—not on worker assignment or processing order.


In [ ]:
import subprocess
import sys

command = [
    sys.executable,
    "nvarc_direct_sampling_runner.py",
    "--config",
    str(config_path),
]
print(" ".join(command))
subprocess.run(command, check=True)


## Aggregate submission and metrics

Aggregation first constructs a complete two-attempt submission for every challenge output. In evaluation mode only, public labels are then loaded to compute both Kaggle-style output-weighted pass@1/pass@2 and the repository's task-averaged metrics.


In [ ]:
from nvarc_direct_sampling import aggregate_run

summary = aggregate_run(config_path)
summary


In [ ]:
# Compact public-evaluation summary; competition reruns have no labels.
if "metrics" in summary:
    metrics = summary["metrics"]
    print(f"Variant: {summary['run_variant']} {summary['attempt_seed_labels']}")
    print(
        "Kaggle-style output-weighted: "
        f"pass@1={metrics['kaggle_output_weighted_pass_at_1']:.4%}, "
        f"pass@2={metrics['kaggle_output_weighted_pass_at_2']:.4%}"
    )
    print(
        "Task-averaged: "
        f"pass@1={metrics['task_averaged_pass_at_1']:.4%}, "
        f"pass@2={metrics['task_averaged_pass_at_2']:.4%}"
    )
    print("Diagnostics:", summary["diagnostics"])
else:
    print(f"Submission written to {summary['submission_path']}")
    print("Diagnostics:", summary["diagnostics"])


## Produced artifacts and interpretation

- `/kaggle/working/submission.json`: exact two-attempt Kaggle submission.
- `<run_dir>/config.json`: selected variant, seed mapping, model paths, limits, and excluded-method protocol.
- `<run_dir>/environment.json`: Python and package versions.
- `<run_dir>/worker_*_manifest.json`: GPU and frozen-checkpoint contracts.
- `<run_dir>/task_results.pbz2`: per-output draws, seeds, validity, dimensions, truncation, and runtime.
- `<run_dir>/evaluation_results.json`: public metrics and diagnostics when labels are available.
- `<run_dir>/run_summary.json`: compact run-level coverage and result summary.

Interpret `S1` as output-weighted pass@1 because A is duplicated. Compare it with `S2_A`, whose first attempt is the identical A draw, to isolate the value of independent draw B. Treat `S2_B` and `S2_C` as predeclared pass@2 replication runs; report the primary run and the replicate range rather than selecting the highest leaderboard score.

The fixed `[[0]]` placeholder is used only because Kaggle requires two syntactically valid grids. It is never treated as a valid generated attempt in public metrics, though Kaggle could theoretically award an accidental match.
